# Statistical Diagnostics Deep Dive

FerroML is designed for practitioners who need **rigorous statistical inference**, not just predictions. This notebook provides a comprehensive guide to the statistical diagnostics available in FerroML.

**What you'll learn:**
1. How to interpret R-style model summaries
2. Understanding coefficient inference (t-tests, p-values, confidence intervals)
3. Model fit assessment (R², adjusted R², F-statistic)
4. Prediction intervals vs. confidence intervals
5. Residual analysis and diagnostics
6. Odds ratios and logistic regression diagnostics
7. Model comparison metrics (AIC, BIC)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

## 1. Why Statistical Diagnostics Matter

Most ML libraries focus only on predictions: fit a model, get predictions, move on. But in many domains, we need to know:

- **Is a coefficient statistically significant?** (p-values, confidence intervals)
- **How much can we trust our predictions?** (prediction intervals)
- **Is the model correctly specified?** (residual diagnostics)
- **How do features affect the outcome?** (effect sizes, odds ratios)

FerroML provides all of this out of the box.

## 2. Linear Regression Deep Dive

Let's create a dataset where we know the true coefficients, then see how well FerroML recovers them with proper inference.

In [ ]:
from ferroml.linear import LinearRegression

# Generate data with known true coefficients
n_samples = 200
n_features = 4

# True coefficients: [3.0, -2.0, 1.5, 0.0]
# Note: Feature 4 has NO effect (coefficient = 0)
true_coef = np.array([3.0, -2.0, 1.5, 0.0])
true_intercept = 5.0
noise_std = 2.0

X = np.random.randn(n_samples, n_features)
y = true_intercept + X @ true_coef + np.random.randn(n_samples) * noise_std

print("True model: y = 5.0 + 3.0*x1 - 2.0*x2 + 1.5*x3 + 0.0*x4 + noise")
print(f"Noise std: {noise_std}")

### 2.1 R-Style Summary Output

FerroML's `summary()` function provides comprehensive output similar to R's `summary(lm(...))`, showing coefficient estimates, standard errors, t-statistics, and p-values.

In [ ]:
# Fit the model
model = LinearRegression()
model.fit(X, y)

# Print R-style summary
print(model.summary())

### Interpreting the Summary Output

The summary shows:

1. **Coefficients Table**: For each coefficient:
   - **Estimate**: The fitted coefficient value
   - **Std Error**: Standard error of the estimate
   - **t-statistic**: Estimate / Std Error (tests if coefficient ≠ 0)
   - **p-value**: Probability of seeing this t-stat if true coefficient = 0

2. **Model Fit Statistics**:
   - **R²**: Proportion of variance explained (0-1)
   - **Adjusted R²**: R² adjusted for number of predictors
   - **F-statistic**: Tests if ALL coefficients are jointly zero

### 2.2 Coefficient Confidence Intervals

P-values tell us *if* a coefficient is significant, but confidence intervals tell us *what range of values* is plausible.

In [ ]:
# Get coefficient CIs at 95% level
coef_info = model.coefficients_with_ci(level=0.95)

print("Coefficient Confidence Intervals (95%)")
print("=" * 70)
print(f"{'Feature':<10} {'Estimate':>10} {'CI Lower':>12} {'CI Upper':>12} {'True':>10} {'Contains?':>10}")
print("-" * 70)

for i, ci in enumerate(coef_info):
    if ci['name'] == 'intercept':
        true_val = true_intercept
    else:
        true_val = true_coef[i-1] if i > 0 else true_coef[i]
    
    contains = "Yes" if ci['ci_lower'] <= true_val <= ci['ci_upper'] else "No"
    print(f"{ci['name']:<10} {ci['estimate']:>10.4f} {ci['ci_lower']:>12.4f} {ci['ci_upper']:>12.4f} {true_val:>10.1f} {contains:>10}")

### Visualizing Confidence Intervals

In [ ]:
# Extract coefficient info (excluding intercept for cleaner visualization)
feature_coefs = [c for c in coef_info if c['name'] != 'intercept']

# Create the plot
fig, ax = plt.subplots(figsize=(10, 6))

names = [c['name'] for c in feature_coefs]
estimates = [c['estimate'] for c in feature_coefs]
ci_lowers = [c['ci_lower'] for c in feature_coefs]
ci_uppers = [c['ci_upper'] for c in feature_coefs]

y_pos = np.arange(len(names))

# Error bars
errors = [[est - lo for est, lo in zip(estimates, ci_lowers)],
          [hi - est for est, hi in zip(estimates, ci_uppers)]]

ax.errorbar(estimates, y_pos, xerr=errors, fmt='o', capsize=5, capthick=2, 
            color='steelblue', markersize=10, label='Estimated (95% CI)')

# True values
ax.scatter(true_coef, y_pos, color='red', marker='x', s=150, linewidths=3, 
           label='True value', zorder=5)

# Zero line
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

ax.set_yticks(y_pos)
ax.set_yticklabels(names)
ax.set_xlabel('Coefficient Value')
ax.set_title('Coefficient Estimates with 95% Confidence Intervals')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key insight**: Notice that Feature 3's confidence interval contains zero, correctly indicating it may not be significantly different from zero (consistent with our true coefficient of 0.0).

### 2.3 Prediction Intervals

**Confidence intervals** capture uncertainty about the mean prediction.
**Prediction intervals** capture uncertainty about individual predictions (including noise).

Prediction intervals are always wider because they account for irreducible error.

In [ ]:
# Generate test points
X_test = np.random.randn(10, n_features)
y_true = true_intercept + X_test @ true_coef + np.random.randn(10) * noise_std

# Get predictions with intervals
y_pred, lower, upper = model.predict_interval(X_test, level=0.95)

print("Predictions with 95% Prediction Intervals")
print("=" * 65)
print(f"{'Sample':>6} {'Predicted':>12} {'Lower':>12} {'Upper':>12} {'True':>12} {'In PI?':>8}")
print("-" * 65)

for i in range(10):
    in_pi = "Yes" if lower[i] <= y_true[i] <= upper[i] else "No"
    print(f"{i+1:>6} {y_pred[i]:>12.3f} {lower[i]:>12.3f} {upper[i]:>12.3f} {y_true[i]:>12.3f} {in_pi:>8}")

coverage = sum(1 for i in range(10) if lower[i] <= y_true[i] <= upper[i]) / 10
print(f"\nEmpirical coverage: {coverage:.1%} (expected: 95%)")

### Visualizing Prediction Intervals

In [ ]:
# Sort by predicted value for cleaner visualization
sort_idx = np.argsort(y_pred)

fig, ax = plt.subplots(figsize=(10, 6))

x_plot = range(len(y_pred))

# Prediction intervals
ax.fill_between(x_plot, lower[sort_idx], upper[sort_idx], alpha=0.3, 
                color='steelblue', label='95% Prediction Interval')

# Predictions
ax.plot(x_plot, y_pred[sort_idx], 'b-', linewidth=2, label='Predicted')

# True values
ax.scatter(x_plot, y_true[sort_idx], color='red', s=80, zorder=5, label='True value')

ax.set_xlabel('Sample (sorted by prediction)')
ax.set_ylabel('y')
ax.set_title('Predictions with 95% Prediction Intervals')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.4 Residual Analysis

Residuals (y - ŷ) should be:
1. **Centered around zero** (no systematic bias)
2. **Constant variance** (homoscedasticity)
3. **Normally distributed** (for valid inference)
4. **Independent** (no autocorrelation)

In [ ]:
# Get residuals and fitted values
residuals = model.residuals()
fitted_values = model.fitted_values()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Fitted
ax1 = axes[0, 0]
ax1.scatter(fitted_values, residuals, alpha=0.6)
ax1.axhline(y=0, color='red', linestyle='--')
ax1.set_xlabel('Fitted Values')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted\n(Check for patterns)')
ax1.grid(True, alpha=0.3)

# 2. Q-Q Plot (normality check)
ax2 = axes[0, 1]
from scipy import stats
stats.probplot(residuals, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot\n(Points should follow diagonal)')
ax2.grid(True, alpha=0.3)

# 3. Histogram of residuals
ax3 = axes[1, 0]
ax3.hist(residuals, bins=20, density=True, alpha=0.7, color='steelblue')
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
ax3.plot(x_norm, stats.norm.pdf(x_norm, residuals.mean(), residuals.std()), 
         'r-', linewidth=2, label='Normal fit')
ax3.set_xlabel('Residuals')
ax3.set_ylabel('Density')
ax3.set_title('Residual Distribution\n(Should be bell-shaped)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Scale-Location plot
ax4 = axes[1, 1]
ax4.scatter(fitted_values, np.sqrt(np.abs(residuals)), alpha=0.6)
ax4.set_xlabel('Fitted Values')
ax4.set_ylabel('sqrt(|Residuals|)')
ax4.set_title('Scale-Location\n(Check for heteroscedasticity)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print residual statistics
print("Residual Statistics:")
print(f"  Mean: {residuals.mean():.6f} (should be ~0)")
print(f"  Std:  {residuals.std():.4f} (compare to noise_std={noise_std})")
print(f"  Min:  {residuals.min():.4f}")
print(f"  Max:  {residuals.max():.4f}")

### 2.5 Model Diagnostics Output

FerroML provides a dedicated `diagnostics()` method for comprehensive model assessment.

In [ ]:
print(model.diagnostics())

## 3. Logistic Regression Deep Dive

Logistic regression has its own set of diagnostics, particularly **odds ratios** which are essential for interpretability in many domains (medicine, social science, etc.).

In [ ]:
from ferroml.linear import LogisticRegression

# Generate binary classification data
n_samples = 500

# Simulating: probability of disease based on age, BMI, and a genetic marker
age = np.random.normal(50, 10, n_samples)  # Age in years
bmi = np.random.normal(25, 5, n_samples)   # Body Mass Index
genetic_marker = np.random.binomial(1, 0.3, n_samples)  # Binary marker

X = np.column_stack([age, bmi, genetic_marker])

# True log-odds:
# log(p/(1-p)) = -10 + 0.1*age + 0.2*bmi + 1.5*genetic
# Meaning: OR(age) = e^0.1 ≈ 1.11, OR(bmi) = e^0.2 ≈ 1.22, OR(genetic) = e^1.5 ≈ 4.48
true_coef_log = np.array([0.1, 0.2, 1.5])
true_intercept_log = -10

log_odds = true_intercept_log + X @ true_coef_log
prob = 1 / (1 + np.exp(-log_odds))
y = np.random.binomial(1, prob).astype(float)

print(f"Generated binary outcome with {y.mean():.1%} positive cases")
print(f"\nTrue odds ratios:")
print(f"  Age:     OR = {np.exp(0.1):.3f} (per year increase)")
print(f"  BMI:     OR = {np.exp(0.2):.3f} (per unit increase)")
print(f"  Genetic: OR = {np.exp(1.5):.3f} (marker present vs absent)")

### 3.1 Logistic Regression Summary

In [ ]:
# Fit logistic regression
lr = LogisticRegression()
lr.fit(X, y)

# Print summary
print(lr.summary())

### 3.2 Odds Ratios

The **odds ratio (OR)** is the exponential of the coefficient: OR = e^β

Interpretation:
- OR > 1: Increased odds of positive outcome
- OR = 1: No effect
- OR < 1: Decreased odds of positive outcome

In [ ]:
# Get odds ratios with confidence intervals
odds_info = lr.odds_ratios_with_ci(level=0.95)

feature_names = ['Age', 'BMI', 'Genetic Marker']
true_ors = [np.exp(0.1), np.exp(0.2), np.exp(1.5)]

print("Odds Ratios with 95% Confidence Intervals")
print("=" * 70)
print(f"{'Feature':<18} {'OR':>10} {'CI Lower':>12} {'CI Upper':>12} {'True OR':>10}")
print("-" * 70)

for i, (oi, name, true_or) in enumerate(zip(odds_info, feature_names, true_ors)):
    print(f"{name:<18} {oi['odds_ratio']:>10.3f} {oi['ci_lower']:>12.3f} {oi['ci_upper']:>12.3f} {true_or:>10.3f}")

### Visualizing Odds Ratios

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ors = [oi['odds_ratio'] for oi in odds_info]
ci_lowers = [oi['ci_lower'] for oi in odds_info]
ci_uppers = [oi['ci_upper'] for oi in odds_info]

y_pos = np.arange(len(feature_names))

# Error bars on log scale
errors = [[or_ - lo for or_, lo in zip(ors, ci_lowers)],
          [hi - or_ for or_, hi in zip(ors, ci_uppers)]]

ax.errorbar(ors, y_pos, xerr=errors, fmt='o', capsize=5, capthick=2,
            color='steelblue', markersize=12, label='Estimated OR (95% CI)')

# True values
ax.scatter(true_ors, y_pos, color='red', marker='x', s=150, linewidths=3,
           label='True OR', zorder=5)

# OR = 1 line (no effect)
ax.axvline(x=1, color='gray', linestyle='--', alpha=0.7, label='OR = 1 (no effect)')

ax.set_yticks(y_pos)
ax.set_yticklabels(feature_names)
ax.set_xlabel('Odds Ratio')
ax.set_title('Odds Ratios with 95% Confidence Intervals')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')  # Log scale is natural for ORs

plt.tight_layout()
plt.show()

**Key insight**: An odds ratio CI that doesn't include 1.0 indicates statistical significance. Notice that all our features have CIs excluding 1.0, indicating they're all significant predictors.

### 3.3 Model Fit and Selection Criteria

In [ ]:
print("Logistic Regression Model Fit Statistics")
print("=" * 50)
print(f"Pseudo R² (McFadden): {lr.pseudo_r_squared():.4f}")
print(f"Log-Likelihood:       {lr.log_likelihood():.2f}")
print(f"AIC:                  {lr.aic():.2f}")
print(f"BIC:                  {lr.bic():.2f}")

lr_stat, lr_pval = lr.likelihood_ratio_test()
print(f"\nLikelihood Ratio Test:")
print(f"  LR Statistic:       {lr_stat:.2f}")
print(f"  p-value:            {lr_pval:.2e}")

### Interpreting These Metrics

1. **Pseudo R² (McFadden)**: Ranges from 0 to 1, but values of 0.2-0.4 are considered excellent for logistic regression

2. **AIC/BIC**: Lower is better. Used for model comparison:
   - AIC: Akaike Information Criterion (-2*LL + 2k)
   - BIC: Bayesian Information Criterion (-2*LL + k*log(n))
   - BIC penalizes complexity more than AIC

3. **Likelihood Ratio Test**: Tests if the model is better than an intercept-only model
   - H0: All coefficients = 0
   - p < 0.05: Model is significantly better than null

### 3.4 Model Comparison: Which Features Matter?

Let's compare models with different subsets of features using AIC/BIC.

In [ ]:
# Compare different models
models = {
    'Full (Age + BMI + Genetic)': X,
    'Age + BMI': X[:, :2],
    'Age + Genetic': X[:, [0, 2]],
    'BMI + Genetic': X[:, [1, 2]],
    'Age only': X[:, [0]],
    'BMI only': X[:, [1]],
    'Genetic only': X[:, [2]],
}

print("Model Comparison using AIC and BIC")
print("=" * 55)
print(f"{'Model':<30} {'AIC':>10} {'BIC':>10}")
print("-" * 55)

results = []
for name, X_subset in models.items():
    m = LogisticRegression()
    m.fit(X_subset, y)
    aic = m.aic()
    bic = m.bic()
    results.append((name, aic, bic))
    print(f"{name:<30} {aic:>10.2f} {bic:>10.2f}")

# Find best model
best_aic = min(results, key=lambda x: x[1])
best_bic = min(results, key=lambda x: x[2])

print(f"\nBest model by AIC: {best_aic[0]}")
print(f"Best model by BIC: {best_bic[0]}")

## 4. Regularized Regression and Inference

**Important**: Standard statistical inference (p-values, CIs) is not valid for regularized models like Ridge and Lasso because regularization biases the estimates.

However, we can still compare model fits and use feature selection properties.

In [ ]:
from ferroml.linear import RidgeRegression, LassoRegression, ElasticNet

# Generate data with some irrelevant features
n_samples = 200
n_informative = 4
n_noise = 6
n_features = n_informative + n_noise

# True coefficients: first 4 are informative, rest are noise
true_coef = np.array([2.0, -1.5, 1.0, -0.5] + [0.0] * n_noise)

X = np.random.randn(n_samples, n_features)
y = X @ true_coef + np.random.randn(n_samples) * 0.5

print(f"Dataset: {n_samples} samples, {n_features} features")
print(f"True informative features: {n_informative}")
print(f"Noise features: {n_noise}")

In [ ]:
# Fit different models
ols = LinearRegression()
ridge = RidgeRegression(alpha=1.0)
lasso = LassoRegression(alpha=0.1)
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)

ols.fit(X, y)
ridge.fit(X, y)
lasso.fit(X, y)
enet.fit(X, y)

# Compare coefficients
print("Coefficient Comparison")
print("=" * 75)
print(f"{'Feature':<10} {'True':>8} {'OLS':>10} {'Ridge':>10} {'Lasso':>10} {'ElasticNet':>12}")
print("-" * 75)

for i in range(n_features):
    info = "*" if i < n_informative else ""
    print(f"x{i}{info:<9} {true_coef[i]:>8.3f} {ols.coef_[i]:>10.4f} {ridge.coef_[i]:>10.4f} {lasso.coef_[i]:>10.4f} {enet.coef_[i]:>12.4f}")

print("\n* = informative feature")

### Key Observations:

1. **OLS**: Estimates all coefficients, may fit noise
2. **Ridge**: Shrinks all coefficients toward zero, but none become exactly zero
3. **Lasso**: Sets many coefficients to exactly zero (sparse), performing feature selection
4. **Elastic Net**: Balance between Ridge (grouping effect) and Lasso (sparsity)

In [ ]:
# Visualize coefficient shrinkage
fig, ax = plt.subplots(figsize=(12, 6))

x_pos = np.arange(n_features)
width = 0.2

ax.bar(x_pos - 1.5*width, true_coef, width, label='True', color='black', alpha=0.8)
ax.bar(x_pos - 0.5*width, ols.coef_, width, label='OLS', color='steelblue')
ax.bar(x_pos + 0.5*width, lasso.coef_, width, label='Lasso', color='coral')
ax.bar(x_pos + 1.5*width, enet.coef_, width, label='Elastic Net', color='seagreen')

ax.axvline(x=n_informative - 0.5, color='red', linestyle='--', alpha=0.5)
ax.text(n_informative - 0.5, ax.get_ylim()[1] * 0.9, '  Noise features ->', fontsize=10, color='red')

ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
ax.set_title('Coefficient Shrinkage: OLS vs Lasso vs Elastic Net')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'x{i}' for i in range(n_features)])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Comparing Model Performance with R²

In [ ]:
print("Model Comparison: R² Values")
print("=" * 40)
print(f"OLS:         R² = {ols.r_squared():.4f}")
print(f"Ridge:       R² = {ridge.r_squared():.4f}")
print(f"Lasso:       R² = {lasso.r_squared():.4f}")
print(f"Elastic Net: R² = {enet.r_squared():.4f}")

print(f"\nNon-zero coefficients:")
print(f"OLS:         {(np.abs(ols.coef_) > 1e-10).sum()} / {n_features}")
print(f"Lasso:       {lasso.n_nonzero()} / {n_features}")
print(f"Elastic Net: {enet.n_nonzero()} / {n_features}")

## 6. Summary: When to Use Which Diagnostics

| Scenario | Recommended Diagnostics |
|----------|-------------------------|
| **Coefficient significance** | p-values, confidence intervals |
| **Effect size interpretation** | Odds ratios (logistic), standardized coefficients |
| **Prediction uncertainty** | Prediction intervals |
| **Model fit quality** | R², adjusted R², residual plots |
| **Model comparison** | AIC, BIC, likelihood ratio test |
| **Feature selection** | Lasso (exact zeros), significance testing |
| **Assumption checking** | Residual diagnostics, Q-Q plots |

### FerroML vs. Other Libraries

| Feature | sklearn | statsmodels | FerroML |
|---------|---------|-------------|----------|
| Coefficient p-values | No | Yes | **Yes** |
| Confidence intervals | No | Yes | **Yes** |
| Prediction intervals | No | Yes | **Yes** |
| Odds ratios with CI | No | Yes | **Yes** |
| R-style summaries | No | Yes | **Yes** |
| Residual diagnostics | No | Yes | **Yes** |
| AIC/BIC | No | Yes | **Yes** |
| sklearn-compatible API | Yes | No | **Yes** |
| Rust performance | No | No | **Yes** |

## Next Steps

- **03_ferroml_vs_sklearn.ipynb**: Detailed performance and API comparison
- **04_production_deployment.ipynb**: ONNX export and deployment

For more information, see the [FerroML documentation](https://github.com/ferroml/ferroml).